In [5]:
# expected output:

# | customer_id | first_name | last_name |
# | ----------- | ---------- | --------- |
# | 1002        | Bob        | Smith     |
# | 1004        | David      | Wilson    |
# | 1007        | Grace      | Moore     |
# | 1009        | Ivy        | Anderson  |
# | 1013        | Mia        | Martin    |
# | 1015        | Olivia     | Garcia    |

import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as func
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder.appName("CustomersWithoutOrders").master("local[*]").getOrCreate()

customerSchema = StructType([
    StructField("customer_id", StringType()),
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("email", StringType()),
    StructField("city", StringType())
])

orderSchema = StructType([
    StructField("order_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("order_date", StringType()),
    StructField("amount", IntegerType())
])

customers = spark.read.csv(
    "./customers.csv",
    schema=customerSchema,
    header=True
)

orders = spark.read.csv(
    "orders.csv",
    schema=orderSchema,
    header=True
)

customersWithoutOrders = (
    customers.join(orders, on=["customer_id"], how="left_anti")
    .select(func.col("customer_id"), func.col("first_name"), func.col("last_name"))
)

customersWithoutOrders.show()

spark.stop()

+-----------+----------+---------+
|customer_id|first_name|last_name|
+-----------+----------+---------+
|       1002|       Bob|    Smith|
|       1004|     David|   Wilson|
|       1007|     Grace|    Moore|
|       1009|       Ivy| Anderson|
|       1013|       Mia|   Martin|
|       1015|    Olivia|   Garcia|
+-----------+----------+---------+

